# 07 — Model-Based RL: Learning $P$ and $R$, Then Planning

Notebook 06 closed on an observation. Every episode of Q-learning saw a
transition — this state, this action, that reward, that next state — used it for
one update, and threw it away. Those observations are exactly what you would need
to *estimate the environment itself*. And notebook 02 already showed that with
$P$ and $R$ in hand, the problem can be solved outright.

So the series has been working with two of the three possibilities and ignoring
the third:

| | Model | Method | Notebook |
| --- | --- | --- | --- |
| **Planning** | handed to you | value iteration | 02 |
| **Model-free learning** | never built | MC, REINFORCE, PPO, TD, Q-learning | 03–06 |
| **Model-based learning** | *estimated from data* | this notebook | 07 |

The idea is almost embarrassingly simple: count what happens, divide, and treat
the result as if it were the truth. That last step has a name —
**certainty equivalence** — and the whole notebook is really about how much
trouble the phrase "as if it were the truth" is hiding.

What follows: the model estimator, a head-to-head against Q-learning, Dyna-Q,
and then a failure that the standard way of measuring success does not detect.

In [1]:
# --- environment from notebook 01, repeated so this notebook stands alone ---
from __future__ import annotations

from enum import IntEnum
from typing import NamedTuple

import numpy as np

class State(IntEnum):
    NO_INFO = 0           # nothing checked yet
    SALINITY_CHECKED = 1  # feed salinity known
    FOULING_CHECKED = 2   # fouling indicators known
    BOTH_CHECKED = 3      # both kinds of evidence in hand
    SUCCESS = 4           # problem solved (terminal)
    FAILURE = 5           # wrong or unsafe fix submitted (terminal)


class Action(IntEnum):
    CHECK_SALINITY = 0
    CHECK_FOULING = 1
    RUN_SIMULATION = 2
    SUBMIT_DIRECTLY = 3


N_STATES, N_ACTIONS = len(State), len(Action)
TERMINAL_STATES = frozenset({State.SUCCESS, State.FAILURE})
NONTERMINAL = [s for s in State if s not in TERMINAL_STATES]


def is_terminal(state) -> bool:
    return State(state) in TERMINAL_STATES


COST_CHECK = -0.5          # first look at a piece of evidence
COST_REPEAT_CHECK = -1.0   # re-checking something already known: pure waste
REWARD_SIM_SUCCESS = 9.0   # +10 outcome, minus the -1 implicit cost of simulating
REWARD_SIM_FAILURE = -11.0
REWARD_SUBMIT_SUCCESS = 10.0
REWARD_SUBMIT_FAILURE = -10.0

SIM_SUCCESS_PROB = {
    State.NO_INFO: 0.15,
    State.SALINITY_CHECKED: 0.55,
    State.FOULING_CHECKED: 0.45,
    State.BOTH_CHECKED: 0.95,
}
SUBMIT_SUCCESS_PROB = {
    State.NO_INFO: 0.05,
    State.SALINITY_CHECKED: 0.35,
    State.FOULING_CHECKED: 0.25,
    State.BOTH_CHECKED: 0.75,
}


class Transition(NamedTuple):
    prob: float
    next_state: State
    reward: float


# Evidence held in each non-terminal state, used to work out where a check lands.
_EVIDENCE = {
    State.NO_INFO: frozenset(),
    State.SALINITY_CHECKED: frozenset({Action.CHECK_SALINITY}),
    State.FOULING_CHECKED: frozenset({Action.CHECK_FOULING}),
    State.BOTH_CHECKED: frozenset({Action.CHECK_SALINITY, Action.CHECK_FOULING}),
}
_STATE_BY_EVIDENCE = {ev: st for st, ev in _EVIDENCE.items()}


def transitions(state, action) -> tuple[Transition, ...]:
    # Every outcome of taking `action` in `state`, probabilities summing to 1.
    state, action = State(state), Action(action)

    if is_terminal(state):
        return (Transition(1.0, state, 0.0),)

    if action in (Action.CHECK_SALINITY, Action.CHECK_FOULING):
        already_known = action in _EVIDENCE[state]
        next_state = (
            state if already_known
            else _STATE_BY_EVIDENCE[_EVIDENCE[state] | {action}]
        )
        reward = COST_REPEAT_CHECK if already_known else COST_CHECK
        return (Transition(1.0, next_state, reward),)

    if action is Action.RUN_SIMULATION:
        p = SIM_SUCCESS_PROB[state]
        return (
            Transition(p, State.SUCCESS, REWARD_SIM_SUCCESS),
            Transition(1.0 - p, State.FAILURE, REWARD_SIM_FAILURE),
        )

    p = SUBMIT_SUCCESS_PROB[state]
    return (
        Transition(p, State.SUCCESS, REWARD_SUBMIT_SUCCESS),
        Transition(1.0 - p, State.FAILURE, REWARD_SUBMIT_FAILURE),
    )


def transition_tables() -> tuple[np.ndarray, np.ndarray]:
    # Dense tables for exact methods: P[s, a, s'] and expected R[s, a].
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        for a in Action:
            for prob, next_state, reward in transitions(s, a):
                P[s, a, next_state] += prob
                R[s, a] += prob * reward
    return P, R


P, R = transition_tables()


def step(state, action, rng) -> tuple[State, float, bool]:
    # Sample one environment step: (next_state, reward, done).
    outcomes = transitions(state, action)
    if len(outcomes) == 1:
        only = outcomes[0]
        return only.next_state, only.reward, is_terminal(only.next_state)
    u, cumulative = rng.random(), 0.0
    for prob, next_state, reward in outcomes:
        cumulative += prob
        if u < cumulative:
            return next_state, reward, is_terminal(next_state)
    last = outcomes[-1]  # float-rounding fallback
    return last.next_state, last.reward, is_terminal(last.next_state)


GAMMA = 0.95

print("setup complete;", N_STATES, "states,", N_ACTIONS, "actions, gamma =", GAMMA)

setup complete; 6 states, 4 actions, gamma = 0.95


## The answer key

Same as ever — notebook 02's exact solution, recomputed so this notebook stands
alone. `policy_value` is the exact policy evaluation from 02: solve
$(I - \gamma P_\pi) v_\pi = R_\pi$ directly. We will need it to score plans
against the truth rather than against their own estimates, which is the entire
methodological point of this notebook.

In [2]:
def value_iteration(P_, R_, gamma=GAMMA, tol=1e-13, max_sweeps=10_000):
    """Bellman optimality iteration on any (P, R) -- true or estimated."""
    V = np.zeros(N_STATES)
    for _ in range(max_sweeps):
        V_next = (R_ + gamma * P_ @ V).max(axis=1)
        V_next[list(TERMINAL_STATES)] = 0.0
        if np.abs(V_next - V).max() < tol:
            return V_next, R_ + gamma * P_ @ V_next
        V = V_next
    raise RuntimeError("value iteration did not converge")


def policy_value(action_of_state, gamma=GAMMA):
    """Exact value of a deterministic policy under the TRUE dynamics."""
    pi = np.zeros((N_STATES, N_ACTIONS))
    for s in NONTERMINAL:
        pi[s, action_of_state(s)] = 1.0
    P_pi = np.einsum("sa,sat->st", pi, P)
    R_pi = np.einsum("sa,sa->s", pi, R)
    keep = [int(s) for s in NONTERMINAL]
    v = np.zeros(N_STATES)
    v[keep] = np.linalg.solve(
        np.eye(len(keep)) - gamma * P_pi[np.ix_(keep, keep)], R_pi[keep]
    )
    return v


V_STAR, Q_STAR = value_iteration(P, R)
OPTIMAL_ACTIONS = {
    State.NO_INFO: {Action.CHECK_SALINITY, Action.CHECK_FOULING},
    State.SALINITY_CHECKED: {Action.CHECK_FOULING},
    State.FOULING_CHECKED: {Action.CHECK_SALINITY},
    State.BOTH_CHECKED: {Action.RUN_SIMULATION},
}
assert np.allclose(V_STAR, [6.245, 7.1, 7.1, 8.0, 0.0, 0.0])

STARTS = [int(s) for s in NONTERMINAL]
print("v* =", np.round(V_STAR[:4], 4))

v* = [6.245 7.1   7.1   8.   ]


## Counting is all it takes

The maximum-likelihood estimate of a tabular MDP is exactly what it sounds like.
For every $(s, a)$ you have tried, $\hat{P}(s' \mid s,a)$ is the fraction of
times you landed in $s'$, and $\hat{R}(s,a)$ is the mean reward you collected.
No optimisation, no gradients — just tallies.

The interesting design decision is what to do about pairs you have **never
tried**, because those have no data to average. Every choice here is a prior in
disguise. The one below assumes an untried action leaves you where you are and
pays `unseen_reward`; we come back to how much that assumption matters.

In [3]:
class Model:
    """Maximum-likelihood tabular model: counts in, P_hat and R_hat out."""

    def __init__(self, unseen_reward=0.0):
        self.n = np.zeros((N_STATES, N_ACTIONS, N_STATES))   # transition counts
        self.r = np.zeros((N_STATES, N_ACTIONS))             # summed rewards
        self.k = np.zeros((N_STATES, N_ACTIONS))             # times tried
        self.unseen_reward = unseen_reward

    def add(self, s, a, r, next_s):
        self.n[s, a, next_s] += 1
        self.r[s, a] += r
        self.k[s, a] += 1

    def tables(self):
        P_hat = np.zeros((N_STATES, N_ACTIONS, N_STATES))
        R_hat = np.zeros((N_STATES, N_ACTIONS))
        for s in range(N_STATES):
            for a in range(N_ACTIONS):
                if self.k[s, a] > 0:
                    P_hat[s, a] = self.n[s, a] / self.k[s, a]
                    R_hat[s, a] = self.r[s, a] / self.k[s, a]
                else:
                    P_hat[s, a, s] = 1.0          # never tried: assume self-loop
                    R_hat[s, a] = self.unseen_reward
        return P_hat, R_hat

    def untried(self):
        return int((self.k[:len(NONTERMINAL)] == 0).sum())


def random_policy(s, rng):
    return int(rng.integers(N_ACTIONS))


rng = np.random.default_rng(0)
m = Model()
s, a = State.BOTH_CHECKED, Action.RUN_SIMULATION

print(f"estimating P(SUCCESS | BOTH_CHECKED, RUN_SIMULATION), true value "
      f"{SIM_SUCCESS_PROB[State.BOTH_CHECKED]}\n")
print(f"{'samples':>9}{'P_hat':>10}{'error':>10}")
for target in [1, 5, 20, 100, 500, 2000]:
    while m.k[s, a] < target:
        ns, r, _ = step(s, a, rng)
        m.add(s, a, r, ns)
    P_hat, R_hat = m.tables()
    p = P_hat[s, a, State.SUCCESS]
    print(f"{target:>9}{p:>10.4f}{p - P[s, a, State.SUCCESS]:>+10.4f}")

estimating P(SUCCESS | BOTH_CHECKED, RUN_SIMULATION), true value 0.95

  samples     P_hat     error
        1    1.0000   +0.0500
        5    1.0000   +0.0500
       20    1.0000   +0.0500
      100    0.9200   -0.0300
      500    0.9380   -0.0120
     2000    0.9500   +0.0000


That is the whole estimator. Now the substantive question: is it worth it?

## Certainty equivalence

Plan on $\hat{P}, \hat{R}$ as though they were $P, R$. After every episode,
rebuild the tables and re-run value iteration — on six states that costs nothing,
and it means the agent always acts on everything it knows.

The behaviour policy is $\varepsilon$-greedy on the *planned* $Q$, with exploring
starts, exactly matching notebook 06's Q-learning so the comparison is about the
algorithm rather than the data collection.

In [4]:
def eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(N_ACTIONS))
    return int(np.argmax(Q[s]))


def certainty_equivalence(n_episodes, seed, eps=0.1, unseen_reward=0.0):
    """Learn the model from experience, re-plan after every episode."""
    rng = np.random.default_rng(seed)
    model = Model(unseen_reward)
    Q = np.zeros((N_STATES, N_ACTIONS))
    for _ in range(n_episodes):
        s = int(rng.choice(STARTS))
        while True:
            a = eps_greedy(Q, s, eps, rng)
            next_s, r, done = step(s, a, rng)
            model.add(s, a, r, next_s)
            if done:
                break
            s = next_s
        _, Q = value_iteration(*model.tables())     # plan on everything so far
    return Q, model


def q_learning(n_episodes, seed, eps=0.1, alpha=0.1):
    """Notebook 06's Q-learning, unchanged."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    for _ in range(n_episodes):
        s = int(rng.choice(STARTS))
        while True:
            a = eps_greedy(Q, s, eps, rng)
            next_s, r, done = step(s, a, rng)
            Q[s, a] += alpha * ((r if done else r + GAMMA * Q[next_s].max()) - Q[s, a])
            if done:
                break
            s = next_s
    return Q


def score(Q):
    """(actions optimal out of 4, TRUE value of the greedy policy from NO_INFO)."""
    picks = {s: Action(int(Q[s].argmax())) for s in NONTERMINAL}
    n_opt = sum(picks[s] in OPTIMAL_ACTIONS[s] for s in NONTERMINAL)
    return n_opt, policy_value(lambda s: picks[State(s)])[State.NO_INFO]


Q_ce, model = certainty_equivalence(20, seed=0)
n_opt, value = score(Q_ce)
print(f"after 20 episodes: {n_opt}/4 optimal, true value {value:.4f} "
      f"(optimal {V_STAR[State.NO_INFO]:.4f})")
print(f"untried state-action pairs: {model.untried()} of {4 * N_ACTIONS}")

after 20 episodes: 4/4 optimal, true value 6.2450 (optimal 6.2450)
untried state-action pairs: 6 of 16


## Head to head

Three methods on identical episode budgets: plan on a learned model, learn values
directly, or do both. Dyna-Q is the "both" — take a real step, update $Q$ from
it, then replay `n_plan` remembered transitions through the model and update
$Q$ from those too. It is model-based sample *reuse* bolted onto a model-free
learner.

In [5]:
def dyna_q(n_episodes, seed, n_plan=5, eps=0.1, alpha=0.1):
    """Q-learning plus n_plan replayed updates from the learned model per step."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    model, seen = Model(), []
    for _ in range(n_episodes):
        s = int(rng.choice(STARTS))
        while True:
            a = eps_greedy(Q, s, eps, rng)
            next_s, r, done = step(s, a, rng)
            model.add(s, a, r, next_s)
            if (s, a) not in seen:
                seen.append((s, a))
            Q[s, a] += alpha * ((r if done else r + GAMMA * Q[next_s].max()) - Q[s, a])

            for _ in range(n_plan):                     # replay from the model
                ps, pa = seen[int(rng.integers(len(seen)))]
                probs = model.n[ps, pa] / model.k[ps, pa]
                p_next = int(rng.choice(N_STATES, p=probs))
                p_r = model.r[ps, pa] / model.k[ps, pa]
                target = p_r if is_terminal(p_next) else p_r + GAMMA * Q[p_next].max()
                Q[ps, pa] += alpha * (target - Q[ps, pa])

            if done:
                break
            s = next_s
    return Q


BUDGETS, SEEDS = [3, 10, 30, 100, 300], 40

print(f"true value of the greedy policy from NO_INFO, mean of {SEEDS} seeds")
print(f"optimal = {V_STAR[State.NO_INFO]:.4f}\n")
print(f"{'episodes':>9}{'certainty eq.':>15}{'Q-learning':>13}{'Dyna-Q (5)':>13}")
for n in BUDGETS:
    ce = np.mean([score(certainty_equivalence(n, s)[0])[1] for s in range(SEEDS)])
    ql = np.mean([score(q_learning(n, s))[1] for s in range(SEEDS)])
    dy = np.mean([score(dyna_q(n, s))[1] for s in range(SEEDS)])
    print(f"{n:>9}{ce:>15.4f}{ql:>13.4f}{dy:>13.4f}")

true value of the greedy policy from NO_INFO, mean of 40 seeds
optimal = 6.2450

 episodes  certainty eq.   Q-learning   Dyna-Q (5)
        3        -3.0947      -2.4789      -1.4724
       10         3.4294      -2.6594       4.5564


       30         5.1264       5.5254       5.7344


      100         5.7581       6.1096       6.1096


      300         6.1096       6.1773       5.6132


**The model earns its keep when data is scarce, and stops mattering once there is
enough of it.** At ten episodes the gap between planning on a learned model and
learning values directly is enormous — Q-learning is still returning a *negative*
policy while the planner is already positive. By three hundred episodes they are
indistinguishable — and the remaining wobble in the columns is seed noise, not
signal. The policy value is a discrete quantity here (there are only so many
deterministic policies), so averaging forty seeds still produces a lumpy curve
rather than a smooth one; the Dyna column dipping at three hundred is that, not a
regression.

The reason is not that the model contains more information — it contains exactly
the same transitions. It is that a model lets one observation affect the *entire*
value function. A single visit to `BOTH_CHECKED` tells Q-learning one number;
told to a model and then planned on, it propagates back through every state that
can reach `BOTH_CHECKED`, immediately. Q-learning has to wait for a trajectory to
carry the information back, one bootstrap at a time.

Dyna-Q sits where you would expect: the replayed updates are precisely a cheap
approximation to re-planning, and they buy most of the early advantage.

## Accuracy or coverage?

The obvious hypothesis is that the plan improves because the model gets more
accurate. It is worth checking, because it is wrong.

In [6]:
print(f"mean of {SEEDS} seeds\n")
print(f"{'episodes':>9}{'mean |P_hat - P|':>18}{'untried (s,a)':>15}"
      f"{'opt/4':>8}{'true value':>12}")
for n in [3, 5, 10, 20, 50, 200]:
    errs, untried, opts, vals = [], [], [], []
    for seed in range(SEEDS):
        Q, model = certainty_equivalence(n, seed)
        P_hat, _ = model.tables()
        seen = model.k > 0
        errs.append(float(np.abs(P_hat - P)[seen].mean()))
        untried.append(model.untried())
        o, v = score(Q)
        opts.append(o); vals.append(v)
    print(f"{n:>9}{np.mean(errs):>18.4f}{np.mean(untried):>15.2f}"
          f"{np.mean(opts):>8.2f}{np.mean(vals):>12.4f}")

mean of 40 seeds

 episodes  mean |P_hat - P|  untried (s,a)   opt/4  true value
        3            0.0296          10.57    3.12     -3.0947
        5            0.0289           9.12    3.35      1.4546
       10            0.0228           7.90    3.40      3.4294
       20            0.0194           7.03    3.55      4.9601
       50            0.0212           5.33    3.67      5.5681


      200            0.0238           2.00    3.90      5.9742


The accuracy of the entries the model *has* barely moves — it hovers around
$0.02$–$0.03$ throughout, because as fast as old entries get better, newly
discovered $(s,a)$ pairs arrive with one sample each and drag the average back.
Meanwhile the policy improves from catastrophic to near-optimal.

What tracks the improvement is the **untried** column. Coverage, not precision.
This is a general and slightly counter-intuitive property of tabular model-based
RL: a crude estimate of a transition you have seen a few times is enough to plan
correctly, whereas a transition you have never seen is not something any amount
of care elsewhere can compensate for.

Which sets up the failure.

## Dyna-Q: how much planning per step?

In [7]:
print(f"true value of the greedy policy, mean of {SEEDS} seeds\n")
print(f"{'n_plan':>7}" + "".join(f"{n:>12}" for n in [3, 10, 30, 100]))
for n_plan in [0, 1, 5, 20]:
    row = [np.mean([score(dyna_q(n, s, n_plan=n_plan))[1] for s in range(SEEDS)])
           for n in [3, 10, 30, 100]]
    tag = f"{n_plan}" + (" (= Q-learning)" if n_plan == 0 else "")
    print(f"{tag:>7}" + "".join(f"{v:>12.4f}" for v in row))

true value of the greedy policy, mean of 40 seeds

 n_plan           3          10          30         100
0 (= Q-learning)     -2.4789     -2.6594      5.5254      6.1096


      1     -3.4549      0.3477      5.8401      4.9815


      5     -1.4724      4.5564      5.7344      6.1096


     20      0.4203      4.1455      4.9031      5.2083


`n_plan = 0` is Q-learning by definition — no replay, no model consulted.

The ten-episode column is where the effect is unambiguous: replay lifts the
result from a negative policy to a clearly positive one, monotonically in
`n_plan`. Elsewhere the picture is messier — at three episodes there is barely a
model to replay from, and at a hundred every setting has effectively converged, so
the ordering is decided by noise. Read this table as "replay helps most exactly
when data is scarcest and the model is already worth consulting", and do not read
a trend into the larger budgets.

The case for Dyna is a wall-clock one anyway: a planning step costs a table
lookup while a real step may cost a robot trial or a plant experiment. Where that
ratio is extreme, spending twenty model updates per real step is obviously worth
it even for the modest gains above.

## The failure: confident about what it never saw

Everything above collected data with $\varepsilon$-greedy exploration and
exploring starts. Suppose instead the data comes from an agent behaving *well* —
running the optimal policy from notebook 02, no exploration at all. Two thousand
episodes of clean, on-policy, expert data. Then plan on it.

In [8]:
def careful(state):
    """The optimal policy from notebook 02."""
    return {
        State.NO_INFO: Action.CHECK_SALINITY,
        State.SALINITY_CHECKED: Action.CHECK_FOULING,
        State.FOULING_CHECKED: Action.CHECK_SALINITY,
        State.BOTH_CHECKED: Action.RUN_SIMULATION,
    }[State(state)]


def collect_on_policy(n_episodes, seed):
    rng = np.random.default_rng(seed)
    model = Model()
    for _ in range(n_episodes):
        s = State.NO_INFO
        while True:
            a = careful(s)
            next_s, r, done = step(s, a, rng)
            model.add(s, a, r, next_s)
            if done:
                break
            s = next_s
    return model


expert = collect_on_policy(2000, seed=0)
_, Q_hat = value_iteration(*expert.tables())

print("2,000 episodes of the OPTIMAL policy, no exploration\n")
print(f"{'state':<19}{'action':<17}{'tried':>7}{'q* (true)':>11}{'q_hat':>9}")
for s in NONTERMINAL:
    for a in Action:
        print(f"{State(s).name:<19}{Action(a).name:<17}"
              f"{int(expert.k[s, a]):>7}{Q_STAR[s, a]:>11.3f}{Q_hat[s, a]:>9.3f}")
print(f"\nq_hat[FOULING_CHECKED] = {np.round(Q_hat[State.FOULING_CHECKED], 3)}"
      "   <- four identical numbers, from zero data")

2,000 episodes of the OPTIMAL policy, no exploration

state              action             tried  q* (true)    q_hat
NO_INFO            CHECK_SALINITY      2000      6.245    6.245
NO_INFO            CHECK_FOULING          0      6.245    5.933
NO_INFO            RUN_SIMULATION         0     -8.000    5.933
NO_INFO            SUBMIT_DIRECTLY        0     -9.000    5.933
SALINITY_CHECKED   CHECK_SALINITY         0      5.745    6.745
SALINITY_CHECKED   CHECK_FOULING       2000      7.100    7.100
SALINITY_CHECKED   RUN_SIMULATION         0      0.000    6.745
SALINITY_CHECKED   SUBMIT_DIRECTLY        0     -3.000    6.745
FOULING_CHECKED    CHECK_SALINITY         0      7.100    0.000
FOULING_CHECKED    CHECK_FOULING          0      5.745    0.000
FOULING_CHECKED    RUN_SIMULATION         0     -2.000    0.000
FOULING_CHECKED    SUBMIT_DIRECTLY        0     -5.000    0.000
BOTH_CHECKED       CHECK_SALINITY         0      6.600    7.600
BOTH_CHECKED       CHECK_FOULING          0      6

Three of the sixteen pairs were ever tried. `FOULING_CHECKED` was never *entered*
— the optimal policy checks salinity first, so that branch of the MDP does not
exist as far as this data is concerned — and all four of its estimated action
values are identical. The planner has no information there whatsoever.

It will still emit a policy for it, with no indication that anything is wrong.
Which action it picks is decided by `argmax`'s tie-breaking rule, and nothing
else.

In [9]:
def evaluate_plan(Q, tie_break="first"):
    if tie_break == "first":
        picks = {s: Action(int(np.argmax(Q[s]))) for s in NONTERMINAL}
    else:                       # identical values, opposite tie-break
        picks = {s: Action(int(N_ACTIONS - 1 - np.argmax(Q[s][::-1])))
                 for s in NONTERMINAL}
    v = policy_value(lambda s: picks[State(s)])
    return picks, v


for tb in ("first", "last"):
    picks, v = evaluate_plan(Q_hat, tb)
    print(f"ties broken toward the {tb} action:")
    print(f"  {'start state':<19}{'planned':<17}{'true value':>12}{'v*':>9}{'gap':>9}")
    for s in NONTERMINAL:
        print(f"  {State(s).name:<19}{picks[s].name:<17}"
              f"{v[s]:>12.3f}{V_STAR[s]:>9.3f}{v[s] - V_STAR[s]:>+9.3f}")
    print(f"  value from NO_INFO only: {v[State.NO_INFO]:.4f}\n")

ties broken toward the first action:
  start state        planned            true value       v*      gap
  NO_INFO            CHECK_SALINITY          6.245    6.245   +0.000
  SALINITY_CHECKED   CHECK_FOULING           7.100    7.100   +0.000
  FOULING_CHECKED    CHECK_SALINITY          7.100    7.100   +0.000
  BOTH_CHECKED       RUN_SIMULATION          8.000    8.000   +0.000
  value from NO_INFO only: 6.2450

ties broken toward the last action:
  start state        planned            true value       v*      gap
  NO_INFO            CHECK_SALINITY          6.245    6.245   +0.000
  SALINITY_CHECKED   CHECK_FOULING           7.100    7.100   +0.000
  FOULING_CHECKED    SUBMIT_DIRECTLY        -5.000    7.100  -12.100
  BOTH_CHECKED       RUN_SIMULATION          8.000    8.000   +0.000
  value from NO_INFO only: 6.2450



Read the last line of each block first. **Measured the way every other experiment
in this notebook measured — the value of the plan from `NO_INFO` — both plans
score $6.245$, exactly optimal.** By that metric nothing is wrong.

Measured from `FOULING_CHECKED`, one of them is worth $-5.0$ against an optimal
$7.1$: a gap of over twelve, from a plan built on two thousand episodes of expert
data, by changing nothing but a tie-breaking convention. The model fit is
byte-identical in both cases.

The start-state metric cannot see it because the optimal policy never routes
through `FOULING_CHECKED`. The plan is catastrophically wrong exactly where it is
never asked, which is precisely the situation that changes the moment anything
perturbs the system — a different initial state, a stochastic outcome that lands
somewhere unusual, a deployment where the first check fails.

Three things worth separating here, because they are usually run together:

1. **The model was not wrong.** Every number it estimated from data was close to
   correct. The problem is the entries it estimated from *nothing*, and the fact
   that it reports those with the same authority as the rest.
2. **Expert data is not good data.** The better the demonstrator, the narrower
   the coverage. Two thousand episodes of the optimal policy tried three of
   sixteen actions; two thousand episodes of a random policy would have tried all
   sixteen. For learning a model, the incompetent agent collects the more useful
   dataset.
3. **A model does not create information.** It reorganises what you have, which
   is why it is so much more sample-efficient in the head-to-head above — and it
   is also why it cannot rescue you from data you never collected. Q-learning
   trained on this same dataset would also know nothing about `FOULING_CHECKED`;
   the difference is that its zeros look like ignorance, while a planner's
   confident policy looks like an answer.

The standard defences are all forms of refusing to trust an unvisited entry:
optimism under uncertainty (R-max initialises unseen pairs to the maximum
possible value, so the planner is *driven* to try them), posterior sampling,
count-based bonuses, or simply reporting coverage alongside the plan. The one
thing that does not work is what the cell above did, which is to plug in a
plausible default and say nothing.

## Where the series stands

| Notebook | Model | What it learns | Cost |
| --- | --- | --- | --- |
| 02 | given | $v^*$ exactly | needs $P$ and $R$ |
| 03 | none | $v_\pi$ from returns | $1/\sqrt{N}$, no control |
| 04–05 | none | a policy, directly | high variance; on-policy |
| 06 | none | $q$, bootstrapped | biased early; needs coverage |
| **07** | **learned** | $\hat{P}, \hat{R}$, then plan | confident where it is blind |

Every one of these is tabular: a lookup entry per state, or per state-action
pair. That is what made all of it checkable against notebook 02's exact answers,
and it is also the assumption that fails first in practice. Six states fit in a
table; a real RO plant's sensor readings do not, and neither does the text an
agent reads. The next thing to break is the table itself — replacing it with a
function that generalises across states it has never seen — which changes every
convergence guarantee in the series.

### Things worth trying

- Set `unseen_reward=-11.0` (pessimism) or `+10.0` (optimism) in
  `certainty_equivalence` and re-run the head-to-head. Optimism should explore
  harder; does it actually help here, and at which budget?
- Give the expert-data planner one episode of random actions and re-run the
  failure. How much exploration does it take to close a twelve-point gap?
- Replace the maximum-likelihood counts with a Dirichlet posterior mean (add one
  imaginary visit to every $(s, a, s')$) and see what it does to both the
  head-to-head and the failure.
- In `dyna_q`, replay the *most recent* transitions instead of uniformly random
  ones. That is prioritised sweeping in its crudest form.